# 00 - Model Server Connection & Response Test

Use this diagnostic notebook to test if your local LLM server (`llama-cpp-python` / local server) is active, responsive, and able to return completion results quickly before running long LLaMEA evolution experiments.

## 1. Setup Environment & Path Definitions

In [ ]:
import os
import sys
import time
import json
from pathlib import Path

# Ensure project root src/ is on sys.path
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
src_path = str(root_dir / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Set default local server variables if not present
os.environ.setdefault("LLM_PROVIDER", "local")
os.environ.setdefault("LOCAL_LLM_BASE_URL", "http://localhost:1234/v1")
os.environ.setdefault("LOCAL_LLM_MODEL", "qwen2.5-coder-1.5b-instruct-q4_k_m")
os.environ.setdefault("LOCAL_LLM_API_KEY", "not-needed")

print(f"Project Root:  {root_dir}")
print(f"Provider:      {os.environ['LLM_PROVIDER']}")
print(f"Endpoint:      {os.environ['LOCAL_LLM_BASE_URL']}")
print(f"Model Name:    {os.environ['LOCAL_LLM_MODEL']}")

## 2. Low-Level HTTP Endpoint Test (Ping & Models List)
Verify that the HTTP endpoint on port 1234 responds to basic requests.

In [ ]:
import urllib.request

base_url = os.environ["LOCAL_LLM_BASE_URL"].rstrip("/")
models_url = f"{base_url}/models"

print(f"Sending HTTP GET request to: {models_url} ...")
try:
    start_t = time.time()
    req = urllib.request.Request(models_url, headers={"User-Agent": "Python-Test"})
    with urllib.request.urlopen(req, timeout=10) as response:
        status = response.status
        body = json.loads(response.read().decode('utf-8'))
        elapsed = time.time() - start_t
    print(f"\n[OK] Server responded in {elapsed:.3f} seconds with status code {status}!")
    print("Available models payload:")
    print(json.dumps(body, indent=2))
except Exception as e:
    print(f"\n[ERROR] Connection failed: {e}")
    print("Make sure your server is running via `bash scripts/03_serve_model.sh`!")

## 3. Direct OpenAI Client Completion Test
Send a short prompt to the model directly using `openai.OpenAI` client and measure generation speed.

In [ ]:
import openai

client = openai.OpenAI(
    base_url=os.environ["LOCAL_LLM_BASE_URL"],
    api_key=os.environ["LOCAL_LLM_API_KEY"]
)

test_prompt = "Write a simple Python function called add(a, b) that returns the sum of a and b."

print(f"Sending prompt to model ('{os.environ['LOCAL_LLM_MODEL']}')...")
print(f"Prompt: '{test_prompt}'\n")

start_time = time.time()
response = client.chat.completions.create(
    model=os.environ["LOCAL_LLM_MODEL"],
    messages=[
        {"role": "system", "content": "You are a helpful coding assistant."},
        {"role": "user", "content": test_prompt}
    ],
    temperature=0.7,
    max_tokens=256
)
duration = time.time() - start_time

result_text = response.choices[0].message.content
usage = response.usage

print("==================== MODEL RESPONSE ====================")
print(result_text)
print("========================================================")
print(f"Time taken:       {duration:.2f} seconds")
if usage:
    print(f"Prompt tokens:    {usage.prompt_tokens}")
    print(f"Completion tokens:{usage.completion_tokens}")
    if usage.completion_tokens and duration > 0:
        print(f"Generation speed: {usage.completion_tokens / duration:.2f} tokens/sec")

## 4. Native LLaMEA Provider Integration Test
Test calling the model through `src.llm.providers.build_llm()` to verify the LLaMEA abstraction layer works seamlessly.

In [ ]:
from llm.providers import build_llm

print("Building LLM instance via build_llm('local')...")
llm = build_llm("local")

prompt = "Create a Python function named hello_world() that prints 'Hello World!'."
messages = [{"role": "user", "content": prompt}]

start_time = time.time()
# Query via LLaMEA's query() interface
response_text = llm.query(messages)
duration = time.time() - start_time

print(f"\n[OK] LLaMEA Provider response received in {duration:.2f}s:")
print("--------------------------------------------------------")
print(response_text)
print("--------------------------------------------------------")